In [2]:
import pandas as pd
import numpy as np
df = pd.read_excel('/content/cases_train2 (12).xlsx')
df['Оценка'] = df['Оценка'].round().astype('Int64')
df_test_cases = pd.read_excel('/content/Test1_2 (3).xlsx')
df_test_cases['Оценка'] = df_test_cases['Оценка'].round().astype('Int64')
#df = pd.read_excel("/content/good table.xlsx")
df.head(100)


,Кейс,Решение,Решение кейса,ЦА,Проработка решения,Финансовая модель и метрики,Анализ рисков,Доказательства,Оценка
0,1,1,Я выбрал отрасль туризма и гостиничного бизнес...,1,1,1,1,1,1
1,1,2,Я выбрал для разработки отраслевого решения сф...,2,2,2,2,2,2
2,1,4,"Я выбрал отрасль образования, а именно сегмент...",2,4,4,3,3,3
3,1,5,Я выбрал отрасль розничной торговли продуктами...,1,3,2,2,1,2
4,1,6,Я выбрал для отраслевого решения сферу гостини...,3,2,2,2,3,2
...,...,...,...,...,...,...,...,...,...
95,1,97,Я выбрал отрасль грузоперевозок. Целевая аудит...,2,2,4,3,1,2
96,1,98,Я выбрал отрасль ремонта техники. Целевая ауди...,3,2,3,2,2,2
97,1,99,Я выбрал отрасль такси. Целевая аудитория: вод...,1,1,4,3,1,2
98,1,100,Я выбрал отрасль ремонта квартир. Целевая ауди...,3,3,3,2,2,3


In [3]:
df["Доказательства"].value_counts()

,count
Доказательства,
2,480
4,420
3,386
1,312
5,303


In [4]:
mask = df["Кейс"] == 10
df[mask]["Оценка"].value_counts()

,count
Оценка,
4,50
3,47
2,30
5,20
1,14


In [5]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.4 MB/s eta 0:00:00


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from catboost import CatBoostRegressor
import pickle

In [7]:
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModel.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
import re
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"https\S+", " ", text)
    text = re.sub(r"www\.\S+", " ", text)
    text = re.sub(r"@\S+", " ", text)
    text = re.sub("[^a-zA-Z0-9]", " ", text)
    text = re.sub(" +", " ", text).strip()
    return text
df["text_clean"] = df["Решение кейса"].apply(clean_text)
df_test_cases["text_clean"] = df_test_cases["Решение кейса"].apply(clean_text)

In [9]:
import torch

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model.to(device)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [11]:
def get_cls_embeddings(text):
  all_cls = []
  bert_model.eval()
  with torch.no_grad():
    for i in range(0, len(text), 16):
      text_batch = text[i:i+16]
      enc = tokenizer(text_batch, padding=True, truncation=True, max_length = 64, return_tensors="pt")
      enc = {k: v.to(device) for k, v in enc.items()}
      out = bert_model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], token_type_ids=enc["token_type_ids"])
      cls = out.last_hidden_state[:, 0, :].cpu().numpy()
      all_cls.append(cls)
  return np.vstack(all_cls)

In [12]:
X_text_bert = df["text_clean"].astype("str").values
X_text_bert_test = df_test_cases["text_clean"].astype("str").values
X_bert_cls_train= get_cls_embeddings(list(X_text_bert))
X_bert_cls_test = get_cls_embeddings(list(X_text_bert_test))

In [13]:
y_audience = df['ЦА'].values
#X_text = df['Решение кейса'].fillna('').values
X_text = df['text_clean'].fillna('').values

X_train_text_audience, X_test_text_audience, y_train_audience, y_test_audience = train_test_split(
    X_text, y_audience, test_size=0.2, random_state=42, stratify=y_audience
)
X_bert_cls_train= get_cls_embeddings(list(X_train_text_audience))
X_bert_cls_test = get_cls_embeddings(list(X_test_text_audience))
tfidf_audience = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=2)
svd_audience = TruncatedSVD(n_components=200, random_state=42)
X_train_tfidf_audience = tfidf_audience.fit_transform(X_train_text_audience)
X_train_svd_audience = np.hstack([X_train_tfidf_audience.toarray(), X_bert_cls_train])
#X_train_svd_audience = svd_audience.fit_transform(X_train_svd_audience)
X_test_tfidf_audience = tfidf_audience.transform(X_test_text_audience)
X_test_svd_audience = np.hstack([X_test_tfidf_audience.toarray(), X_bert_cls_test])
#X_test_svd_audience = svd_audience.transform(X_test_svd_audience)

In [14]:
from sklearn.model_selection import GridSearchCV

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [16]:
from catboost import CatBoostClassifier

In [17]:
model_audience = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=3,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    auto_class_weights='Balanced',
    early_stopping_rounds=50,
    verbose=50,
    random_seed=42
)

In [18]:
model_audience.fit(X_train_svd_audience , y_train_audience)

0:	learn: 1.5454255	total: 215ms	remaining: 3m 34s
50:	learn: 1.1334815	total: 7.16s	remaining: 2m 13s
100:	learn: 1.0373172	total: 15.4s	remaining: 2m 17s
150:	learn: 0.9527125	total: 22.1s	remaining: 2m 4s
200:	learn: 0.8894497	total: 27.7s	remaining: 1m 50s
250:	learn: 0.8320811	total: 34.5s	remaining: 1m 42s
300:	learn: 0.7905390	total: 40s	remaining: 1m 32s
350:	learn: 0.7494634	total: 46.7s	remaining: 1m 26s
400:	learn: 0.7192997	total: 52.2s	remaining: 1m 18s
450:	learn: 0.6911071	total: 1m 3s	remaining: 1m 17s
500:	learn: 0.6651648	total: 1m 9s	remaining: 1m 8s
550:	learn: 0.6451062	total: 1m 15s	remaining: 1m 1s
600:	learn: 0.6248254	total: 1m 20s	remaining: 53.7s
650:	learn: 0.6040967	total: 1m 27s	remaining: 46.9s
700:	learn: 0.5878008	total: 1m 33s	remaining: 39.7s
750:	learn: 0.5703928	total: 1m 39s	remaining: 33s
800:	learn: 0.5559809	total: 1m 45s	remaining: 26.2s
850:	learn: 0.5393524	total: 1m 51s	remaining: 19.6s
900:	learn: 0.5268473	total: 1m 57s	remaining: 12.9s
95

CatBoostClassifier(auto_class_weights='Balanced', depth=3, early_stopping_rounds=50, eval_metric='MultiClass', iterations=1000, learning_rate=0.1, loss_function='MultiClass', random_seed=42, verbose=50)

In [19]:
y_pred_audience = model_audience.predict(X_test_svd_audience)
print(classification_report(y_test_audience, y_pred_audience))
print("MAE=",mean_absolute_error(y_test_audience, y_pred_audience))

              precision    recall  f1-score   support

           1       0.53      0.63      0.57        49
           2       0.51      0.43      0.46        84
           3       0.48      0.45      0.46        85
           4       0.50      0.58      0.54        88
           5       0.80      0.76      0.78        75

    accuracy                           0.56       381
   macro avg       0.56      0.57      0.56       381
weighted avg       0.56      0.56      0.56       381

MAE= 0.6876640419947506


In [20]:
y_pred_audience_train = model_audience.predict(X_train_svd_audience)
print(classification_report(y_train_audience, y_pred_audience_train))
print("MAE=",mean_absolute_error(y_train_audience, y_pred_audience_train))

              precision    recall  f1-score   support

           1       0.82      0.99      0.90       195
           2       0.95      0.89      0.92       337
           3       0.97      0.94      0.96       338
           4       0.93      0.94      0.94       352
           5       0.98      0.93      0.95       298

    accuracy                           0.93      1520
   macro avg       0.93      0.94      0.93      1520
weighted avg       0.94      0.93      0.94      1520

MAE= 0.11578947368421053


In [21]:
y_sol= df['Проработка решения'].values
X_text = df['text_clean'].fillna('').values
X_train_text_sol, X_test_text_sol, y_train_sol, y_test_sol = train_test_split(
    X_text, y_sol, test_size=0.2, random_state=42
)
X_bert_cls_train= get_cls_embeddings(list(X_train_text_sol))
X_bert_cls_test = get_cls_embeddings(list(X_test_text_sol))
tfidf_sol = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=2)
svd_sol= TruncatedSVD(n_components=200, random_state=42)
X_train_tfidf_sol = tfidf_sol.fit_transform(X_train_text_sol)
X_train_svd_sol = np.hstack([X_train_tfidf_sol.toarray(), X_bert_cls_train])
X_test_tfidf_sol = tfidf_sol.transform(X_test_text_sol)
X_test_svd_sol = np.hstack([X_test_tfidf_sol.toarray(), X_bert_cls_test])

In [22]:
model_sol = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=3,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    auto_class_weights='Balanced',
    early_stopping_rounds=50,
    verbose=50,
    random_seed=42
)
model_sol.fit(X_train_svd_sol , y_train_sol)
y_pred_sol = model_sol.predict(X_test_svd_sol)
#y_pred_sol
print(classification_report(y_test_sol, y_pred_sol))
print("MAE=",mean_absolute_error(y_test_sol, y_pred_sol))

0:	learn: 1.5751802	total: 318ms	remaining: 5m 17s
50:	learn: 1.2179807	total: 11s	remaining: 3m 24s
100:	learn: 1.1151196	total: 16.8s	remaining: 2m 29s
150:	learn: 1.0269952	total: 23.5s	remaining: 2m 12s
200:	learn: 0.9603152	total: 29.1s	remaining: 1m 55s
250:	learn: 0.9044597	total: 35.7s	remaining: 1m 46s
300:	learn: 0.8573916	total: 41.3s	remaining: 1m 35s
350:	learn: 0.8181300	total: 47.7s	remaining: 1m 28s
400:	learn: 0.7804952	total: 53.3s	remaining: 1m 19s
450:	learn: 0.7485740	total: 59.1s	remaining: 1m 11s
500:	learn: 0.7181627	total: 1m 7s	remaining: 1m 7s
550:	learn: 0.6929588	total: 1m 14s	remaining: 1m
600:	learn: 0.6646781	total: 1m 19s	remaining: 53s
650:	learn: 0.6425288	total: 1m 26s	remaining: 46.2s
700:	learn: 0.6258019	total: 1m 31s	remaining: 39.2s
750:	learn: 0.6048995	total: 1m 37s	remaining: 32.3s
800:	learn: 0.5845183	total: 1m 43s	remaining: 25.8s
850:	learn: 0.5681961	total: 1m 49s	remaining: 19.1s
900:	learn: 0.5521189	total: 1m 58s	remaining: 13s
950:	l

In [23]:
y_pred_sol_train = model_sol.predict(X_train_svd_sol)
#y_pred_sol
print(classification_report(y_train_sol, y_pred_sol_train))
print("MAE=",mean_absolute_error(y_train_sol, y_pred_sol_train))

              precision    recall  f1-score   support

           1       0.81      1.00      0.90       185
           2       0.96      0.89      0.92       363
           3       0.97      0.96      0.96       351
           4       0.96      0.95      0.96       338
           5       0.97      0.94      0.96       283

    accuracy                           0.94      1520
   macro avg       0.93      0.95      0.94      1520
weighted avg       0.95      0.94      0.94      1520

MAE= 0.10723684210526316


In [24]:
y_finance= df['Финансовая модель и метрики'].values
X_text = df['text_clean'].fillna('').values
X_train_text_finance, X_test_text_finance,y_train_finance, y_test_finance = train_test_split(
    X_text, y_finance, test_size=0.2, random_state=42
)
X_bert_cls_train= get_cls_embeddings(list(X_train_text_finance))
X_bert_cls_test = get_cls_embeddings(list(X_test_text_finance))

tfidf_finance = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=2)
svd_finance = TruncatedSVD(n_components=200, random_state=42)
X_train_tfidf_finance = tfidf_finance.fit_transform(X_train_text_finance)
X_train_svd_finance = np.hstack([X_train_tfidf_finance.toarray(), X_bert_cls_train])
X_test_tfidf_finance = tfidf_finance.transform(X_test_text_finance)
X_test_svd_finance = np.hstack([X_test_tfidf_finance.toarray(), X_bert_cls_test])

In [25]:
model_finance = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=3,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    auto_class_weights='Balanced',
    early_stopping_rounds=50,
    verbose=50,
    random_seed=42
)
model_finance.fit(X_train_svd_finance , y_train_finance)
y_pred_finance = model_finance.predict(X_test_svd_finance)
print(classification_report(y_test_finance, y_pred_finance))
print("MAE=",mean_absolute_error(y_test_finance, y_pred_finance))

0:	learn: 1.5671477	total: 172ms	remaining: 2m 51s
50:	learn: 1.1195708	total: 5.95s	remaining: 1m 50s
100:	learn: 0.9865104	total: 12.7s	remaining: 1m 53s
150:	learn: 0.8858978	total: 18.4s	remaining: 1m 43s
200:	learn: 0.8180654	total: 24.5s	remaining: 1m 37s
250:	learn: 0.7575904	total: 30.7s	remaining: 1m 31s
300:	learn: 0.7131260	total: 36.7s	remaining: 1m 25s
350:	learn: 0.6790421	total: 43.1s	remaining: 1m 19s
400:	learn: 0.6500709	total: 48.7s	remaining: 1m 12s
450:	learn: 0.6215787	total: 55.4s	remaining: 1m 7s
500:	learn: 0.5960412	total: 1m 3s	remaining: 1m 3s
550:	learn: 0.5707651	total: 1m 10s	remaining: 57.2s
600:	learn: 0.5527891	total: 1m 16s	remaining: 50.7s
650:	learn: 0.5325974	total: 1m 22s	remaining: 44.2s
700:	learn: 0.5150504	total: 1m 28s	remaining: 37.6s
750:	learn: 0.4979718	total: 1m 34s	remaining: 31.4s
800:	learn: 0.4825913	total: 1m 40s	remaining: 24.9s
850:	learn: 0.4665828	total: 1m 47s	remaining: 18.7s
900:	learn: 0.4513292	total: 1m 52s	remaining: 12.4

In [26]:
y_pred_finance_train = model_finance.predict(X_train_svd_finance)
print(classification_report(y_train_finance, y_pred_finance_train))
print("MAE=",mean_absolute_error(y_train_finance, y_pred_finance_train))

              precision    recall  f1-score   support

           1       0.89      1.00      0.94       204
           2       0.99      0.92      0.95       389
           3       0.98      0.95      0.96       384
           4       0.95      0.97      0.96       344
           5       0.98      0.99      0.99       199

    accuracy                           0.96      1520
   macro avg       0.96      0.97      0.96      1520
weighted avg       0.96      0.96      0.96      1520

MAE= 0.06184210526315789


In [27]:
y_risks= df['Анализ рисков'].values
X_text = df['text_clean'].fillna('').values
X_train_text_risks, X_test_text_risks, y_train_risks, y_test_risks = train_test_split(
    X_text, y_risks, test_size=0.2, random_state=42
)
X_bert_cls_train= get_cls_embeddings(list(X_train_text_risks))
X_bert_cls_test = get_cls_embeddings(list(X_test_text_risks))
tfidf_risks = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=2)
svd_risks = TruncatedSVD(n_components=200, random_state=42)
X_train_tfidf_risks = tfidf_risks.fit_transform(X_train_text_risks)
X_train_svd_risks = np.hstack([X_train_tfidf_risks.toarray(), X_bert_cls_train])
X_test_tfidf_risks = tfidf_risks.transform(X_test_text_risks)
X_test_svd_risks = np.hstack([X_test_tfidf_risks.toarray(), X_bert_cls_test])

In [28]:
model_risks = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=3,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    auto_class_weights='Balanced',
    early_stopping_rounds=50,
    verbose=50,
    random_seed=42
)
model_risks.fit(X_train_svd_risks , y_train_risks)
y_pred_risks = model_risks.predict(X_test_svd_risks)
print(classification_report(y_test_risks, y_pred_risks))
print("MAE=",mean_absolute_error(y_test_risks, y_pred_risks))

0:	learn: 1.5780977	total: 158ms	remaining: 2m 37s
50:	learn: 1.2154633	total: 7.13s	remaining: 2m 12s
100:	learn: 1.0964641	total: 12.8s	remaining: 1m 54s
150:	learn: 0.9966493	total: 19.8s	remaining: 1m 51s
200:	learn: 0.9217563	total: 27.9s	remaining: 1m 50s
250:	learn: 0.8625018	total: 34.7s	remaining: 1m 43s
300:	learn: 0.8171787	total: 40.3s	remaining: 1m 33s
350:	learn: 0.7777508	total: 47s	remaining: 1m 26s
400:	learn: 0.7364127	total: 52.7s	remaining: 1m 18s
450:	learn: 0.7084419	total: 59.5s	remaining: 1m 12s
500:	learn: 0.6813112	total: 1m 4s	remaining: 1m 4s
550:	learn: 0.6586094	total: 1m 11s	remaining: 58.4s
600:	learn: 0.6308017	total: 1m 17s	remaining: 51.3s
650:	learn: 0.6075928	total: 1m 24s	remaining: 45.1s
700:	learn: 0.5888763	total: 1m 29s	remaining: 38.2s
750:	learn: 0.5716190	total: 1m 36s	remaining: 31.9s
800:	learn: 0.5535167	total: 1m 41s	remaining: 25.3s
850:	learn: 0.5363777	total: 1m 47s	remaining: 18.9s
900:	learn: 0.5182576	total: 1m 54s	remaining: 12.5s

In [29]:
y_pred_risks_train = model_risks.predict(X_train_svd_risks)
print(classification_report(y_train_risks, y_pred_risks_train))
print("MAE=",mean_absolute_error(y_train_risks, y_pred_risks_train))

              precision    recall  f1-score   support

           1       0.88      1.00      0.93       240
           2       0.97      0.92      0.94       380
           3       0.99      0.93      0.96       374
           4       0.96      0.98      0.97       330
           5       0.97      0.99      0.98       196

    accuracy                           0.96      1520
   macro avg       0.95      0.96      0.96      1520
weighted avg       0.96      0.96      0.96      1520

MAE= 0.06973684210526315


In [30]:
y_proves= df['Доказательства'].values
X_text = X_text = df['text_clean'].fillna('').values
X_train_text_proves, X_test_text_proves, y_train_proves, y_test_proves = train_test_split(
    X_text, y_proves, test_size=0.2, random_state=42
)
X_bert_cls_train= get_cls_embeddings(list(X_train_text_proves))
X_bert_cls_test = get_cls_embeddings(list(X_test_text_proves))
tfidf_proves = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=2)
svd_proves = TruncatedSVD(n_components=200, random_state=42)
X_train_tfidf_proves = tfidf_proves.fit_transform(X_train_text_proves)
X_train_svd_proves = np.hstack([X_train_tfidf_proves.toarray(), X_bert_cls_train])
X_test_tfidf_proves = tfidf_proves.transform(X_test_text_proves)
X_test_svd_proves = np.hstack([X_test_tfidf_proves.toarray(), X_bert_cls_test])

In [31]:
model_proves = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=3,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    auto_class_weights='Balanced',
    early_stopping_rounds=50,
    verbose=50,
    random_seed=42
)
model_proves.fit(X_train_svd_proves , y_train_proves)
y_pred_proves = model_proves.predict(X_test_svd_proves)
print(classification_report(y_test_proves, y_pred_proves))
print("MAE=",mean_absolute_error(y_test_proves, y_pred_proves))

0:	learn: 1.5797316	total: 119ms	remaining: 1m 58s
50:	learn: 1.2082064	total: 5.84s	remaining: 1m 48s
100:	learn: 1.0966096	total: 12.7s	remaining: 1m 53s
150:	learn: 1.0011998	total: 18.3s	remaining: 1m 42s
200:	learn: 0.9211476	total: 25.1s	remaining: 1m 39s
250:	learn: 0.8631921	total: 30.7s	remaining: 1m 31s
300:	learn: 0.8119009	total: 37.4s	remaining: 1m 26s
350:	learn: 0.7719882	total: 42.8s	remaining: 1m 19s
400:	learn: 0.7328208	total: 49.7s	remaining: 1m 14s
450:	learn: 0.7041485	total: 55.3s	remaining: 1m 7s
500:	learn: 0.6774379	total: 1m 1s	remaining: 1m 1s
550:	learn: 0.6471503	total: 1m 7s	remaining: 54.9s
600:	learn: 0.6180339	total: 1m 14s	remaining: 49.1s
650:	learn: 0.5989830	total: 1m 19s	remaining: 42.7s
700:	learn: 0.5773939	total: 1m 25s	remaining: 36.6s
750:	learn: 0.5567012	total: 1m 31s	remaining: 30.4s
800:	learn: 0.5371986	total: 1m 37s	remaining: 24.3s
850:	learn: 0.5206163	total: 1m 43s	remaining: 18.2s
900:	learn: 0.5039359	total: 1m 49s	remaining: 12s
9

In [32]:
y_pred_proves_train = model_proves.predict(X_train_svd_proves)
print(classification_report(y_train_proves, y_pred_proves_train))
print("MAE=",mean_absolute_error(y_train_proves, y_pred_proves_train))

              precision    recall  f1-score   support

           1       0.91      0.99      0.95       238
           2       0.97      0.92      0.94       387
           3       0.99      0.98      0.98       306
           4       0.97      0.97      0.97       346
           5       0.96      0.97      0.97       243

    accuracy                           0.96      1520
   macro avg       0.96      0.97      0.96      1520
weighted avg       0.96      0.96      0.96      1520

MAE= 0.07828947368421052


In [ ]:
new_results_df = pd.DataFrame(columns=['Id','Текст_решения','Анализ ЦА','Проработка решения','Финансовая модель и метрики','Анализ рынков','Доказательства','Предсказанная_оценка','Дата'])

In [ ]:
import re

In [ ]:
dec = '''МОЙ ПРОЕКТ

Я хочу сделать бота для школьников. Он будет помогать решать задачи.

Целевая аудитория - школьники. Им это нужно для учебы.

Мое решение - бот в телеграме. Он будет бесплатный. Похожих ботов нет.

Финансы: разработка стоит примерно 500 тысяч рублей. Потом будем зарабатывать на рекламе.

Риски: могут появиться конкуренты. Будем делать лучше.

Доказательства: я сам учился в школе и знаю, что это нужно. Многие мои друзья тоже так думают.'''

In [33]:
def get_prediction_audience(text):

  if pd.isna(text) or not isinstance(text, str):
        text = ""
  X_bert_cls_test = get_cls_embeddings([text])
  text_tfidf = tfidf_audience.transform([text])
  text_svd = np.hstack([text_tfidf.toarray(), X_bert_cls_test])
  score = model_audience.predict(text_svd)[0]
  return int(score)

In [34]:
def get_prediction_solution(text):
  if pd.isna(text) or not isinstance(text, str):
        text = ""
  X_bert_cls_test = get_cls_embeddings([text])
  text_tfidf = tfidf_sol.transform([text])
  text_svd = np.hstack([text_tfidf.toarray(), X_bert_cls_test])
  score = model_sol.predict(text_svd)[0]
  return int(score)

In [35]:
def get_prediction_finance(text):

  if pd.isna(text) or not isinstance(text, str):
        text = ""
  X_bert_cls_test = get_cls_embeddings([text])
  text_tfidf = tfidf_finance.transform([text])
  text_svd = np.hstack([text_tfidf.toarray(), X_bert_cls_test])
  score = model_finance.predict(text_svd)[0]
  return int(score)

In [36]:
def get_prediction_risks(text):

  if pd.isna(text) or not isinstance(text, str):
        text = ""
  X_bert_cls_test = get_cls_embeddings([text])
  text_tfidf = tfidf_risks.transform([text])
  text_svd = np.hstack([text_tfidf.toarray(), X_bert_cls_test])
  score = model_risks.predict(text_svd)[0]
  return int(score)

In [37]:
def get_prediction_proves(text):

  if pd.isna(text) or not isinstance(text, str):
        text = ""
  X_bert_cls_test = get_cls_embeddings([text])
  text_tfidf = tfidf_proves.transform([text])
  text_svd = np.hstack([text_tfidf.toarray(), X_bert_cls_test])
  score = model_proves.predict(text_svd)[0]
  return int(score)

In [38]:
df_test_cases['pred_audience'] = df_test_cases['text_clean'].apply(get_prediction_audience)
df_test_cases['pred_sol'] = df_test_cases['Решение кейса'].apply(get_prediction_solution)
df_test_cases['pred_finance'] = df_test_cases['Решение кейса'].apply(get_prediction_finance)
df_test_cases['pred_risks'] = df_test_cases['Решение кейса'].apply(get_prediction_risks)
df_test_cases['pred_proves'] = df_test_cases['Решение кейса'].apply(get_prediction_proves)

/tmp/ipykernel_1155/1732985192.py:9: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)
/tmp/ipykernel_1155/3806182561.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)
/tmp/ipykernel_1155/3989254459.py:9: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)
/tmp/ipykernel_1155/2769898772.py:9: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract 

In [39]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score
)


In [40]:
#ЦА
accuracy_audience = accuracy_score(df_test_cases['ЦА'], df_test_cases['pred_audience'])
f1micro_audience = f1_score(df_test_cases['ЦА'], df_test_cases['pred_audience'], average='micro')
f1macro_audience = f1_score(df_test_cases['ЦА'], df_test_cases['pred_audience'], average='macro')
print('accuracy_audience= ', accuracy_audience)
print('f1micro_audience= ', f1micro_audience)
print('f1macro_audience= ', f1macro_audience)
print(classification_report(df_test_cases['ЦА'], df_test_cases['pred_audience']))
print("MAE=", mean_absolute_error(df_test_cases['ЦА'], df_test_cases['pred_audience']))

accuracy_audience=  0.4153846153846154
f1micro_audience=  0.4153846153846154
f1macro_audience=  0.40925680189831126
              precision    recall  f1-score   support

           1       0.35      0.74      0.47        23
           2       0.23      0.12      0.15        26
           3       0.53      0.29      0.38        34
           4       0.32      0.50      0.39        22
           5       0.87      0.52      0.65        25

    accuracy                           0.42       130
   macro avg       0.46      0.43      0.41       130
weighted avg       0.47      0.42      0.40       130

MAE= 0.9153846153846154


In [41]:
accuracy_sol = accuracy_score(df_test_cases['Проработка решения'], df_test_cases['pred_sol'])
f1micro_sol = f1_score(df_test_cases['Проработка решения'], df_test_cases['pred_sol'], average='micro')
f1macro_sol = f1_score(df_test_cases['Проработка решения'], df_test_cases['pred_sol'], average='macro')
print('accuracy_sol= ', accuracy_sol)
print('f1micro_sol= ', f1micro_sol)
print('f1macro_sol= ', f1macro_sol)
print(classification_report(df_test_cases['Проработка решения'], df_test_cases['pred_sol']))
print("MAE=", mean_absolute_error(df_test_cases['Проработка решения'], df_test_cases['pred_sol']))

accuracy_sol=  0.16923076923076924
f1micro_sol=  0.16923076923076924
f1macro_sol=  0.058278145695364235
              precision    recall  f1-score   support

           1       0.00      0.00      0.00        21
           2       0.00      0.00      0.00        32
           3       0.00      0.00      0.00        30
           4       0.00      0.00      0.00        25
           5       0.17      1.00      0.29        22

    accuracy                           0.17       130
   macro avg       0.03      0.20      0.06       130
weighted avg       0.03      0.17      0.05       130

MAE= 2.0307692307692307


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [42]:
accuracy_finance = accuracy_score(df_test_cases['Финансовая модель'], df_test_cases['pred_finance'])
f1micro_finance = f1_score(df_test_cases['Финансовая модель'], df_test_cases['pred_finance'], average='micro')
f1macro_finance = f1_score(df_test_cases['Финансовая модель'], df_test_cases['pred_finance'], average='macro')
print('accuracy_finance= ', accuracy_finance)
print('f1micro_finance= ', f1micro_finance)
print('f1macro_finance= ', f1macro_finance)
print(classification_report(df_test_cases['Финансовая модель'], df_test_cases['pred_finance']))
print("MAE=", mean_absolute_error(df_test_cases['Финансовая модель'], df_test_cases['pred_finance']))

accuracy_finance=  0.24615384615384617
f1micro_finance=  0.24615384615384617
f1macro_finance=  0.13548387096774195
              precision    recall  f1-score   support

           1       0.00      0.00      0.00        23
           2       0.34      0.30      0.32        33
           3       0.00      0.00      0.00        31
           4       0.22      0.92      0.35        24
           5       0.00      0.00      0.00        19

    accuracy                           0.25       130
   macro avg       0.11      0.24      0.14       130
weighted avg       0.13      0.25      0.15       130

MAE= 1.2923076923076924


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
accuracy_risks = accuracy_score(df_test_cases['Анализ рынков'], df_test_cases['pred_risks'])
f1micro_risks = f1_score(df_test_cases['Анализ рынков'], df_test_cases['pred_risks'], average='micro')
f1macro_risks = f1_score(df_test_cases['Анализ рынков'], df_test_cases['pred_risks'], average='macro')
print('accuracy_risks= ', accuracy_risks)
print('f1micro_risks= ', f1micro_risks)
print('f1macro_risks= ', f1macro_risks)
print(classification_report(df_test_cases['Анализ рынков'], df_test_cases['pred_risks']))
print("MAE=", mean_absolute_error(df_test_cases['Анализ рынков'], df_test_cases['pred_risks']))

accuracy_risks=  0.27692307692307694
f1micro_risks=  0.27692307692307694
f1macro_risks=  0.15033384694401644
              precision    recall  f1-score   support

           1       0.00      0.00      0.00        26
           2       0.41      0.34      0.37        32
           3       0.25      0.83      0.38        30
           4       0.00      0.00      0.00        27
           5       0.00      0.00      0.00        15

    accuracy                           0.28       130
   macro avg       0.13      0.24      0.15       130
weighted avg       0.16      0.28      0.18       130

MAE= 1.1


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [43]:
accuracy_proves = accuracy_score(df_test_cases['Доказательства'], df_test_cases['pred_proves'])
f1micro_proves = f1_score(df_test_cases['Доказательства'], df_test_cases['pred_proves'], average='micro')
f1macro_proves = f1_score(df_test_cases['Доказательства'], df_test_cases['pred_proves'], average='macro')
print('accuracy_proves= ', accuracy_proves)
print('f1micro_proves= ', f1micro_proves)
print('f1macro_proves= ', f1macro_proves)
print(classification_report(df_test_cases['Доказательства'], df_test_cases['pred_proves']))
print("MAE", mean_absolute_error(df_test_cases['Доказательства'], df_test_cases['pred_proves']))


accuracy_proves=  0.23846153846153847
f1micro_proves=  0.23846153846153847
f1macro_proves=  0.18244778807438652
              precision    recall  f1-score   support

           1       0.00      0.00      0.00        18
           2       0.32      0.21      0.25        38
           3       0.22      0.08      0.12        25
           4       0.19      0.26      0.22        19
           5       0.23      0.53      0.32        30

    accuracy                           0.24       130
   macro avg       0.19      0.22      0.18       130
weighted avg       0.22      0.24      0.20       130

MAE 1.5461538461538462


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
sec_dec = """АНАЛИЗ ЦА:
Выделено 3 сегмента:
- Школьники 10-11 класс (45%)
- Студенты (35%)
- Учителя (20%)
Проведен опрос 100 человек.

РЕШЕНИЕ:
Экосистема из 2 продуктов: телеграм бот + дашборд для учителей.
Партнеры: Яндекс.Образование, МФТИ.

ФИНАНСЫ:
CAPEX: 2 млн руб. CAC = 100 руб, LTV = 2000 руб.
Окупаемость: 10 месяцев.

РИСКИ:
Технические (30%) - резервный сервер.

ДОКАЗАТЕЛЬСТВА:
Пилот в школе: NPS = 65.
"""

In [44]:
df_test_cases["Predict_score"] = round((df_test_cases['pred_audience'] + df_test_cases['pred_sol'] + df_test_cases['pred_finance'] + df_test_cases['pred_risks'] + df_test_cases['pred_proves'])/5)

In [45]:
print("MAE= ",mean_absolute_error(df_test_cases["Оценка"], df_test_cases["Predict_score"]))

MAE=  1.0307692307692307


In [ ]:
sol_test = """ Я выбрала отрасль кофеен и кофе-точек с собой (кофе на вынос, кофейные киоски, небольшие кофейни на 2–5 столиков). ЦА разделена на две группы. Первая группа — это микробизнес: кофе-байки и кофе-островки с одним сотрудником, работают как ИП или самозанятые. Их проблема в том, что кофейные зерна нужно покупать свежей обжарки каждую неделю, а хорошие обжарщики требуют предоплату 100% за партию от 5 кг, это около 20–30 тысяч рублей единовременно, что для маленькой точки с ежедневной выручкой 5–7 тысяч рублей чувствительно. Вторая группа — это малый бизнес: кофейни с посадочными местами и штатом 2–5 бариста. Их проблема в том, что сезонность спроса сильно различается: зимой продажи выше за счет горячих напитков, а летом люди покупают холодный кофе и чаще берут с собой, но в межсезонье (апрель и октябрь) падение выручки достигает 30%, при этом аренду и зарплату платить надо. Я опиралась на данные исследования «Рынок кофеен России 2024» от компании CoffeeData: рост рынка на 15% за год, количество кофеен достигло 12 тысяч, из них 70% — малый и микробизнес. Также я посмотрел открытую статистику по кофейному рынку на сайте Росстата и несколько статей в профильных телеграм-каналах. В качестве решения я предлагаю продукт «Альфа.Кофе»: кредит на закупку зёрен с отсрочкой первого платежа на 30 дней и с льготным периодом 0% на первые две недели, а также сезонный овердрафт на покрытие аренды в межсезонье на сумму до 100 тысяч рублей. Партнеры — два крупных обжарщика зерна «Coffe Lab» и «Torrefacto», с которыми можно договориться о более выгодных ценах для клиентов банка. Отличие от конкурентов в том, что ни у Сбера, ни у Т-Банка нет специального продукта для кофеен с отсрочкой именно под зерно. По финансам: разработка кредитного продукта обойдется примерно в 6 миллионов рублей, интеграция с партнерами-обжарщиками — в 2 миллиона, маркетинг в кофейных чатах и через конференции бариста — в 2 миллиона. Прогноз: 350 клиентов в первый год, средний доход с клиента — 20 тысяч рублей (за счет процентов по кредиту и эквайринга), выручка — 7 миллионов рублей. Окупаемость — примерно 2,5 года. Риски: конкурентный — другие банки могут запустить похожие продукты; кредитный — часть клиентов может не вернуть деньги, но мы будем проверять по выписке с кофемашины, кто сколько продает. В доказательство я использовал данные CoffeeData и результаты пары интервью с владельцами кофеен в своем городе."""

In [ ]:
predicted_score_audience = get_prediction_audience(sec_dec)
predicted_score_sol = get_prediction_solution(sec_dec)
predicted_score_finance = get_prediction_finance(sec_dec)
predicted_score_risks = get_prediction_risks(sec_dec)
predicted_score_proves = get_prediction_proves(sec_dec)
predicted_score_audience2 = get_prediction_audience(sec_dec)
predicted_score_sol2 = get_prediction_solution(sec_dec)
predicted_score_finance2 = get_prediction_finance(sec_dec)
predicted_score_risks2 = get_prediction_risks(sec_dec)
predicted_score_proves2 = get_prediction_proves(sec_dec)
new_row = pd.DataFrame([{
    'Id': 1,
    'Текст_решения': sec_dec,
    'Анализ ЦА': predicted_score_audience,
    'Проработка решения': predicted_score_sol,
    'Финансовая модель и метрики': predicted_score_finance,
    'Анализ рынков': predicted_score_risks,
    'Доказательства': predicted_score_proves,
    'Предсказанная_оценка': round((predicted_score_audience + predicted_score_sol +
                      predicted_score_finance + predicted_score_risks +
                      predicted_score_proves) / 5),
    'Дата': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
}])
new_row2 = pd.DataFrame([{
    'Id': 1,
    'Текст_решения': sec_dec,
    'Анализ ЦА': predicted_score_audience2,
    'Проработка решения': predicted_score_sol2,
    'Финансовая модель и метрики': predicted_score_finance2,
    'Анализ рынков': predicted_score_risks2,
    'Доказательства': predicted_score_proves2,
    'Предсказанная_оценка': round((predicted_score_audience2 + predicted_score_sol2 +
                      predicted_score_finance2 + predicted_score_risks2 +
                      predicted_score_proves2) / 5),
    'Дата': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
}])
new_results_df = pd.concat([new_results_df, new_row, new_row2], ignore_index=True)
new_results_df.to_excel("новая_таблица.xlsx", index=False)


/tmp/ipykernel_3876/1005490930.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)
/tmp/ipykernel_3876/1097642392.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)


In [ ]:
new_results_df

,Id,Текст_решения,Анализ ЦА,Проработка решения,Финансовая модель и метрики,Анализ рынков,Доказательства,Предсказанная_оценка,Дата
0,1,АНАЛИЗ ЦА:\nВыделено 3 сегмента:\n- Школьники ...,5,4,4,4,4,4,2026-06-22 15:34:44
1,1,АНАЛИЗ ЦА:\nВыделено 3 сегмента:\n- Школьники ...,5,4,4,4,4,4,2026-06-22 15:34:44
